# agentic_alpha_canary_sprint_search_v1

Diagnostic R-G-E-V-D card for canary momentum, LETF compounding, and short-hold sprint hypotheses. It uses qlib `D.features` from the SOXL/SOXX/VIX provider, evaluates canary/dip/guarded sprint candidates, and closes the branch against the production `qlib_paper_sim_v1` return and MaxDD gate.

External seeds: 45ck/llm-quant sprint-alpha notes, arXiv leveraged ETF compounding, volatility-managed portfolios, leverage moving-average risk management, defensive/canary asset allocation, and a Twitter/X leveraged ETF risk seed.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'SPEC.md').exists() and (path / 'src').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

from src.factor import (
    build_canary_sprint_frame,
    compact_canary_sprint_summary,
    evaluate_canary_sprint_candidates,
)
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_canary_sprint_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data/qlib/soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production/qlib_paper_sim_v1'
FEATURE_START = '2010-03-11'
FEATURE_END = '2026-06-10'
TEST_START = '2019-01-02'
TEST_END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
CARD.mkdir(parents=True, exist_ok=True)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
qlib.init(
    provider_uri=PROVIDER_URI.as_posix(),
    region='us',
    expression_cache=None,
    dataset_cache=None,
    kernels=1,
)
qlib_frame = D.features(
    ['SOXL', 'SOXX'],
    ['$close', '$volume', '$vix'],
    start_time=FEATURE_START,
    end_time=FEATURE_END,
    freq='day',
)
features = build_canary_sprint_frame(qlib_frame, start=TEST_START, end=TEST_END)
candidate_metrics = evaluate_canary_sprint_candidates(
    features,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
compact_summary = compact_canary_sprint_summary(
    candidate_metrics,
    branches=sorted(candidate_metrics['branch'].unique()),
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
print(json.dumps({'rows': len(candidate_metrics), **compact_summary}, indent=2, sort_keys=True))


[61745:MainThread](2026-06-23 18:28:56,767) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61745:MainThread](2026-06-23 18:28:56,769) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61745:MainThread](2026-06-23 18:28:56,769) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{
  "best_candidate": null,
  "branch_results": {
    "canary_sprint_short_hold": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "trend60_thr0.06_vix26.0_positive_h3_s0.5",
      "best_drawdown_gate_max_drawdown_pct": -17.423822168063097,
      "best_drawdown_gate_return_pct": 542.6166088552307,
      "best_return_candidate": "trend20_thr-0.03_vix26.0_positive_h3_s3.0",
      "best_return_max_drawdown_pct": -97.88024490089981,
      "best_return_pct": 182964.55772896123,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 5250,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "dip_sprint_short_hold": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "dip2_thr-0.08_trend60_vix22.0_h1_s1.0",
      "best_drawdown_gate_max_drawdown_pct": -18.96937144995311,
      "best_drawdown_gate_return_pct": 585.5915

In [3]:
external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': '45ck/llm-quant sprint alpha notes',
            'url': 'https://github.com/45ck/llm-quant',
            'extracted_signal': 'Leveraged ETF sprint branches use VIX crash filters and maximum short holding periods.',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'LETF compounding improves in momentum regimes and deteriorates in mean-reverting regimes.',
        },
        {
            'source_type': 'paper',
            'title': 'Volatility-Managed Portfolios',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431',
            'extracted_signal': 'Reduce exposure when volatility is high because volatility timing can improve risk-adjusted returns.',
        },
        {
            'source_type': 'paper',
            'title': 'Leverage for the Long Run',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2741701',
            'extracted_signal': 'Use leverage when moving-average environments imply lower volatility and positive return streaks.',
        },
        {
            'source_type': 'paper',
            'title': 'Breadth Momentum and the Canary Universe: Defensive Asset Allocation',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3212862',
            'extracted_signal': 'A separate canary universe can govern crash-protection cash fractions.',
        },
        {
            'source_type': 'twitter',
            'title': 'Leveraged ETF risk seed',
            'url': 'https://x.com/WealthCoachMak/status/2060793363373457722',
            'extracted_signal': 'Treat leveraged ETFs as components of a broader system rather than a standalone trade.',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)
candidate_metrics.to_csv(CARD / 'agentic_alpha_canary_sprint_candidate_metrics.csv', index=False)
failure_modes = (
    '# Failure Modes\n\n'
    '- canary_sprint_short_hold: short holding and VIX filters found high-return tails, but high-return settings still crossed crash-level drawdowns; drawdown-gated settings scaled down below the production baseline.\n'
    '- dip_sprint_short_hold: dip entries produced the best drawdown-gated sprint return, but the return remained far below the production strategy.\n'
    '- guarded_sprint_budget: stateful drawdown and volatility budget controls reduced risk, but the remaining exposure budget was too small to beat the production baseline.\n'
)
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')


542

In [4]:
workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
provider_uri: {PROVIDER_URI.as_posix()}
recorder_uri: {RECORDER_URI.as_posix()}
market: all
region: us
segments:
  feature_start: {FEATURE_START}
  feature_end: {FEATURE_END}
  test_start: {TEST_START}
  test_end: {TEST_END}
workflow:
  stage_order:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  baseline_return_pct: {baseline.return_pct}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  workflow_config: workflow.yaml
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')


737

In [5]:
search_summary = {
    'workflow': 'agentic_alpha_canary_sprint_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'goal_achieved': bool(compact_summary['goal_achieved']),
    'best_candidate': compact_summary['best_candidate'],
    'passes_both_count': int(compact_summary['passes_both_count']),
    'branch_results': compact_summary['branch_results'],
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'canary_sprint': {
        'qlib_provider': PROVIDER_URI.as_posix(),
        'qlib_fields': ['$close', '$volume', '$vix'],
        'qlib_feature_rows': int(len(qlib_frame)),
        'feature_rows': int(len(features)),
        'candidate_coverage_start': features['datetime'].min().date().isoformat(),
        'candidate_coverage_end': features['datetime'].max().date().isoformat(),
        'source_coverage_start': features.attrs['source_coverage_start'],
        'source_coverage_end': features.attrs['source_coverage_end'],
        'selected_features': list(features.attrs['source_fields']),
        'cost_per_turnover': 0.0002,
    },
}
(CARD / 'agentic_alpha_canary_sprint_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')


3219

In [6]:
R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(
        provider_uri=PROVIDER_URI.as_posix(),
        feature_start=FEATURE_START,
        feature_end=FEATURE_END,
        test_start=TEST_START,
        test_end=TEST_END,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        baseline=json.dumps(search_summary['baseline'], sort_keys=True),
        candidate_rows=int(len(candidate_metrics)),
    )
    recorder.log_metrics(
        passes_both_count=float(search_summary['passes_both_count']),
        best_canary_return_pct=float(search_summary['branch_results']['canary_sprint_short_hold']['best_return_pct']),
        best_dip_drawdown_gate_return_pct=float(search_summary['branch_results']['dip_sprint_short_hold']['best_drawdown_gate_return_pct']),
    )
    for path in [
        CARD / 'agentic_alpha_canary_sprint_candidate_metrics.csv',
        CARD / 'agentic_alpha_canary_sprint_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_canary_sprint')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.uri).resolve()

search_summary['recorder_id'] = recorder_id
search_summary['recorder_artifacts'] = sorted(
    path.relative_to(recorder_local_dir).as_posix()
    for path in recorder_local_dir.rglob('*')
    if path.is_file()
)
(CARD / 'agentic_alpha_canary_sprint_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': search_summary['passes_both_count']}, indent=2, sort_keys=True))


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[61745:MainThread](2026-06-23 18:29:00,603) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_canary_sprint_search_v1.


[61745:MainThread](2026-06-23 18:29:00,606) INFO - qlib.workflow - [exp.py:258] - Experiment 792729988849016516 starts running ...


[61745:MainThread](2026-06-23 18:29:00,737) INFO - qlib.workflow - [recorder.py:345] - Recorder f4bc3298eea142ba9b0b2fb3de6614cf starts running under Experiment 792729988849016516 ...


[61745:MainThread](2026-06-23 18:29:00,810) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "passes_both_count": 0,
  "recorder_id": "f4bc3298eea142ba9b0b2fb3de6614cf"
}


In [7]:
artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': [
        'agentic_alpha_canary_sprint_candidate_metrics.csv',
        'agentic_alpha_canary_sprint_search_summary.json',
        'external_sources.csv',
        'failure_modes.md',
    ],
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'signals': 'artifacts/signals.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
    },
    'reproducible_from_recorder': True,
}
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'as_of': TEST_END,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'data_range': [TEST_START, TEST_END],
    'feature_warmup_range': [FEATURE_START, FEATURE_END],
    'branch_results': search_summary['branch_results'],
    'search_report': search_summary,
    'canary_sprint': search_summary['canary_sprint'],
    'external_source_counts': search_summary['external_source_counts'],
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
    },
    'workflow_config': 'workflow.yaml',
    'artifact_policy': artifact_policy,
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_canary_sprint_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_canary_sprint_search_summary.json',
    },
    'recorder_artifacts': search_summary['recorder_artifacts'],
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
conclusion = (
    f'# 结论\n\n'
    f'- what ran: 使用 qlib.init + D.features 从 {PROVIDER_URI.as_posix()} 读取 SOXL/SOXX/VIX，测试 canary sprint、dip sprint、guarded sprint budget 三个 R-G-E-V-D 分支。\n'
    f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 本卡 candidate_rows={len(candidate_metrics)}, passes_both_count={search_summary["passes_both_count"]}。\n'
    f'- regime/window differences: 短持有与正自相关过滤能制造高收益尾部，但高收益候选仍穿透 -20% MaxDD；drawdown-gated 候选收益远低于 baseline。\n'
    f'- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    f'- risks: 该分支依赖 SOXX canary 与 SOXL 自身短期状态，无法证明外部 rate/credit lead-lag 对 SOXL 有足够强的因果收益。\n'
    f'- next_action: 继续寻找非同源外部 canary 或更强的日内/事件驱动信号；recorder_id={recorder_id}。\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(search_summary['passes_both_count']), 'branch_results': search_summary['branch_results']}, indent=2, sort_keys=True))


{
  "branch_results": {
    "canary_sprint_short_hold": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "trend60_thr0.06_vix26.0_positive_h3_s0.5",
      "best_drawdown_gate_max_drawdown_pct": -17.423822168063097,
      "best_drawdown_gate_return_pct": 542.6166088552307,
      "best_return_candidate": "trend20_thr-0.03_vix26.0_positive_h3_s3.0",
      "best_return_max_drawdown_pct": -97.88024490089981,
      "best_return_pct": 182964.55772896123,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 5250,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "dip_sprint_short_hold": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "dip2_thr-0.08_trend60_vix22.0_h1_s1.0",
      "best_drawdown_gate_max_drawdown_pct": -18.96937144995311,
      "best_drawdown_gate_return_pct": 585.5915809628458,
      "best_ret